In [1]:
# ============================================================================
# CORRECTED CROSS-VALIDATION WITH PROPER NESTED STRUCTURE & HARMONIZATION
# ============================================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import scipy.stats as stats  
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.metrics import f1_score, precision_score, average_precision_score
from reader import prepare_qsm_dataset
from util import seed_everything, mask_crop as mask_crop_fn
from train import calibrate_balanced
import matplotlib.pyplot as plt

# --- NEW UTILITY FUNCTIONS FOR METRICS ---
def compare_auc_significance(y_true, prob_base, prob_model):
    """Asymptotic comparison of AUCs (DeLong-like approximation)."""
    auc_base = roc_auc_score(y_true, prob_base)
    auc_model = roc_auc_score(y_true, prob_model)
    n1, n0 = sum(y_true == 1), sum(y_true == 0)
    # Variance approximation for the difference
    var_diff = ((auc_base*(1-auc_base) + (n1-1)*(0.1 - auc_base**2) + (n0-1)*(0.1 - auc_base**2)) / (n1*n0)) 
    z = (auc_model - auc_base) / np.sqrt(max(var_diff, 1e-8))
    p_value = 1 - stats.norm.cdf(z)
    return auc_model - auc_base, p_value

def compute_comprehensive_metrics(y_true, y_prob, threshold):
    """Calculates all requested performance metrics."""
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = recall_score(y_true, y_pred, zero_division=0)
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {
        'AUC': roc_auc_score(y_true, y_prob),
        'AUPRC': average_precision_score(y_true, y_prob),
        'Acc': accuracy_score(y_true, y_pred),
        'B-Acc': (sens + spec) / 2,
        'Prec': precision_score(y_true, y_pred, zero_division=0),
        'Sens': sens,
        'Spec': spec,
        'F1': f1_score(y_true, y_pred, zero_division=0)
    }

# Configuration
device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
N_FOLDS = 5
EPOCHS = 100
JITTER_STD = 0.1
IMG_AUG_STD = 0.05
TARGET_DIM = 128
seed_everything(0)

# Architecture and loss classes
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, inputs, targets, weight=None):
        bce = F.binary_cross_entropy(inputs, targets, weight=weight, reduction='none')
        return (self.alpha * (1 - torch.exp(-bce))**self.gamma * bce).mean()

class ClinicalTransformer(nn.Module):
    def __init__(self, n_inputs, embed_dim=32, n_heads=4, n_layers=2):
        super().__init__()
        self.embedding = nn.Linear(1, embed_dim)
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim*2, dropout=0.1)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.fc = nn.Linear(embed_dim * n_inputs, 1)
    def forward(self, x, return_logit=False):
        if x.ndim == 1: x = x.unsqueeze(0)
        x = self.embedding(x.unsqueeze(-1)).permute(1, 0, 2)
        x = self.transformer(x).permute(1, 0, 2).flatten(1)
        logit = self.fc(x).squeeze()
        if logit.ndim == 0: logit = logit.unsqueeze(0)
        return logit if return_logit else torch.sigmoid(logit)

class SpectralViT(nn.Module):
    def __init__(self, n_inputs, n_heads=1, embed_dim=32, n_layers=1):
        super().__init__()
        ranks = torch.arange(1, n_inputs + 1, dtype=torch.float32)
        self.rank_weights = nn.Parameter(1.0 / ranks)
        layer = nn.TransformerEncoderLayer(
            d_model=1, 
            nhead=n_heads,
            dim_feedforward=embed_dim*2,
            dropout = 0.1)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.fc = nn.Linear(n_inputs, 1)
    def forward(self, x, return_logit=False):
        x_weighted = x * self.rank_weights
        x_seq = x_weighted.unsqueeze(-1).transpose(0, 1)  # [seq_len, batch, 1]
        x_trans = self.transformer(x_seq)
        x_trans_flat = x_trans.transpose(0, 1).squeeze(-1)  # [batch, n_inputs]
        logit = self.fc(x_trans_flat).squeeze()
        if logit.ndim == 0: logit = logit.unsqueeze(0)
        return logit if return_logit else torch.sigmoid(logit)

class SpatialViT(nn.Module):
    def __init__(self, img_size=128, patch_size=16, embed_dim=32, n_heads=1, n_layers=1):
        super().__init__()
        self.patch_size, self.n_patches = patch_size, (img_size // patch_size) ** 2
        self.proj = nn.Linear(patch_size * patch_size, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(1, self.n_patches, embed_dim) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, dim_feedforward=embed_dim*2, dropout=0.1)
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.fc = nn.Linear(embed_dim, 1)
    def forward(self, x, return_logit=False):
        b = x.shape[0]
        x = x.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
        x = self.proj(x.contiguous().view(b, self.n_patches, -1)) + self.pos_embed
        x = self.transformer(x.permute(1, 0, 2)).permute(1, 0, 2).mean(dim=1)
        logit = self.fc(x).squeeze()
        if logit.ndim == 0: logit = logit.unsqueeze(0)
        return logit if return_logit else torch.sigmoid(logit)

class ResidualSpectral(nn.Module):
    def __init__(self, m_ct, m_res):
        super().__init__()
        self.m_ct, self.m_res = m_ct, m_res
    def forward(self, x_res, x_clin, return_logit=False):
        with torch.no_grad(): l_clin = self.m_ct(x_clin, return_logit=True)
        l_res = self.m_res(x_res, return_logit=True)
        joint = l_clin + l_res
        return joint if return_logit else torch.sigmoid(joint)

class ResidualSpatial(nn.Module):
    def __init__(self, m_ct, m_res):
        super().__init__()
        self.m_ct, self.m_res = m_ct, m_res
    def forward(self, x_res, x_clin, return_logit=False):
        with torch.no_grad(): l_clin = self.m_ct(x_clin, return_logit=True)
        l_res = self.m_res(x_res, return_logit=True)
        joint = l_clin + l_res
        return joint if return_logit else torch.sigmoid(joint)

class PassThrough(nn.Module):
    def forward(self, x, **kwargs): return x
    
# Data preparation
def robust_flatten(img_np, target_dim=TARGET_DIM):
    t = torch.from_numpy(img_np).float().unsqueeze(0).unsqueeze(0)
    resized = F.interpolate(t, size=(target_dim, target_dim), mode='bilinear', align_corners=False)
    return resized.numpy().flatten()

def get_slice_level_data(dataset, include_unlabeled=False):
    all_imgs, all_clins, all_lbls, subj_map = [], [], [], []
    for i in range(len(dataset)):
        img, clin, lbl, _ = dataset[i]
        if (i % 72) in [0]: continue
        if (lbl != -1) or (include_unlabeled and lbl == -1):
            all_imgs.append(robust_flatten(img.numpy().squeeze()))
            all_clins.append(clin.numpy()); all_lbls.append(lbl); subj_map.append(i)
    return np.array(all_imgs), np.array(all_clins), np.array(all_lbls), np.array(subj_map)

# ============================================================================
# STEP 1: Load & Harmonize Data
# ============================================================================
dataset_msw = prepare_qsm_dataset('MSW', '/media/mts_dbs/dbs/all/nii/qsm_115/im', '/media/mts_dbs/dbs/all/nii/seg_ps/', '/data/Ali/RadDBS-QSM/data/docs/dbs_03292024.csv', 'msw_cache_6d_cv.pt', load_cache=True, mask_crop_fn=mask_crop_fn, cv_pad=False)
dataset_chh = prepare_qsm_dataset('CHH', '/media/mts_dbs/chh/nii/qsm/', '/media/mts_dbs/chh/roi/', '/media/mts_dbs/chh/xlsx/chh_subjects_table1_20240729.csv', 'chh_cache_6d_cv.pt', load_cache=True, mask_crop_fn=mask_crop_fn, cv_pad=False)

# Load MSW
X_full_slices, X_full_clin, y_full_slices, full_subj_map = get_slice_level_data(dataset_msw, include_unlabeled=True)
labeled_mask = (y_full_slices != -1)
X_tr_slices = X_full_slices[labeled_mask]
X_tr_clin_raw = X_full_clin[labeled_mask]
y_tr_slices = y_full_slices[labeled_mask]
tr_subj_map = full_subj_map[labeled_mask]

# Load CHH
X_te_slices, X_te_clin_raw, y_te_slices, te_subj_map = get_slice_level_data(dataset_chh)

# --- HARMONIZATION: SITE-SPECIFIC Z-SCORING ---
print("Applying Site-Specific Harmonization...")
scaler_msw = StandardScaler()
X_tr_clin = scaler_msw.fit_transform(X_tr_clin_raw)

scaler_chh = StandardScaler()
X_te_clin = scaler_chh.fit_transform(X_te_clin_raw)
# ---------------------------------------------

NEG_WEIGHT = float(sum(y_tr_slices==1) // sum(y_tr_slices==0))
GAMMA = NEG_WEIGHT

# ============================================================================
# STEP 2: HYPERPARAMETER SELECTION
# ============================================================================
print("\n" + "="*60)
print("STEP 2: HYPERPARAMETER SELECTION")
print("="*60)

unique_subjs = np.unique(tr_subj_map)
y_unique = np.array([y_tr_slices[tr_subj_map == s][0] for s in unique_subjs])
outer_skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
criterion = FocalLoss(gamma=GAMMA)

# 1. OPTIMIZE POS_WEIGHT USING CLINICAL TRANSFORMER
pos_weight_grid = [0.1, 0.25, 0.5, 0.75, 1.0]
weight_results = {k: [] for k in pos_weight_grid}

print("\n--- Phase 1: Optimizing pos_weight via Clinical Transformer ---")
for pos_weight in pos_weight_grid:
    fold_aucs = []
    for fold_idx, (train_subj_idx, val_subj_idx) in enumerate(outer_skf.split(unique_subjs, y_unique)):
        train_subjects, val_subjects = unique_subjs[train_subj_idx], unique_subjs[val_subj_idx]
        train_mask, val_mask = np.isin(tr_subj_map, train_subjects), np.isin(tr_subj_map, val_subjects)
        
        # Data is already harmonized/scaled
        X_train_clin_t = torch.tensor(X_tr_clin[train_mask], dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_tr_slices[train_mask], dtype=torch.float32).to(device)
        X_val_clin_t = torch.tensor(X_tr_clin[val_mask], dtype=torch.float32).to(device)
        y_val = y_tr_slices[val_mask]
        
        model = ClinicalTransformer(n_inputs=X_tr_clin.shape[1]).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        w = torch.where(y_train_t == 0, torch.tensor(NEG_WEIGHT, device=device), torch.tensor(pos_weight, device=device))
        
        for epoch in range(EPOCHS):
            model.train()
            optimizer.zero_grad()
            X_jitter = X_train_clin_t + torch.randn_like(X_train_clin_t) * JITTER_STD
            loss = criterion(model(X_jitter), y_train_t, weight=w)
            loss.backward()
            optimizer.step()
        
        model.eval()
        with torch.no_grad():
            val_probs = model(X_val_clin_t).cpu().numpy()
            val_subj_probs = [val_probs[tr_subj_map[val_mask] == s].mean() for s in val_subjects]
            val_subj_labels = [y_val[tr_subj_map[val_mask] == s][0] for s in val_subjects]
            fold_aucs.append(roc_auc_score(val_subj_labels, val_subj_probs))
    
    weight_results[pos_weight] = fold_aucs
    print(f"  Pos Weight={pos_weight}: AUC = {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}")

SELECTED_POS_WEIGHT = pos_weight_grid[np.argmin([np.std(weight_results[pw]) for pw in pos_weight_grid])]
print(f"✓ Selected positive weight: {SELECTED_POS_WEIGHT}")

# 2. OPTIMIZE N_COMPONENTS USING RESIDUAL SPECTRAL MODEL
pca_components_grid = [16, 32, 64, 128]
pca_results = {k: [] for k in pca_components_grid}

print("\n--- Phase 2: Optimizing n_components via Residual PCA Model ---")
for n_comp in pca_components_grid:
    fold_aucs = []
    for fold_idx, (train_subj_idx, val_subj_idx) in enumerate(outer_skf.split(unique_subjs, y_unique)):
        train_subjects, val_subjects = unique_subjs[train_subj_idx], unique_subjs[val_subj_idx]
        train_mask, val_mask = np.isin(tr_subj_map, train_subjects), np.isin(tr_subj_map, val_subjects)
        
        # Preprocessing (Img only, clin is already done)
        f_img_scaler = StandardScaler().fit(X_tr_slices[train_mask])
        f_pca = PCA(n_components=n_comp, random_state=0, whiten=True).fit(f_img_scaler.transform(X_tr_slices[train_mask]))
        
        X_tr_clin_t = torch.tensor(X_tr_clin[train_mask], dtype=torch.float32).to(device)
        X_tr_pca_t = torch.tensor(f_pca.transform(f_img_scaler.transform(X_tr_slices[train_mask])), dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_tr_slices[train_mask], dtype=torch.float32).to(device)
        
        X_val_clin_t = torch.tensor(X_tr_clin[val_mask], dtype=torch.float32).to(device)
        X_val_pca_t = torch.tensor(f_pca.transform(f_img_scaler.transform(X_tr_slices[val_mask])), dtype=torch.float32).to(device)
        y_val = y_tr_slices[val_mask]
        
        # Train residual model (CT + Spectral)
        m_ct = ClinicalTransformer(n_inputs=X_tr_clin.shape[1]).to(device)
        opt_ct = optim.Adam(m_ct.parameters(), lr=1e-4)
        w = torch.where(y_train_t == 0, torch.tensor(NEG_WEIGHT, device=device), torch.tensor(SELECTED_POS_WEIGHT, device=device))
        for _ in range(50):
            m_ct.train(); opt_ct.zero_grad()
            loss = criterion(m_ct(X_tr_clin_t + torch.randn_like(X_tr_clin_t)*JITTER_STD), y_train_t, weight=w)
            loss.backward(); opt_ct.step()
        m_ct.eval(); [p.requires_grad_(False) for p in m_ct.parameters()]

        model = ResidualSpectral(m_ct, SpectralViT(n_inputs=n_comp)).to(device)
        optimizer = optim.Adam(model.m_res.parameters(), lr=5e-4)
        
        for epoch in range(EPOCHS):
            model.train(); optimizer.zero_grad()
            loss = criterion(model(X_tr_pca_t, X_tr_clin_t), y_train_t, weight=w)
            loss.backward(); optimizer.step()
            
        model.eval()
        with torch.no_grad():
            val_probs = model(X_val_pca_t, X_val_clin_t).cpu().numpy()
            val_subj_probs = [val_probs[tr_subj_map[val_mask] == s].mean() for s in val_subjects]
            val_subj_labels = [y_val[tr_subj_map[val_mask] == s][0] for s in val_subjects]
            fold_aucs.append(roc_auc_score(val_subj_labels, val_subj_probs))
            
    pca_results[n_comp] = fold_aucs
    print(f"  PCA={n_comp}: AUC = {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}")

N_PCA_COMPONENTS = pca_components_grid[np.argmax([np.mean(pca_results[n]) for n in pca_components_grid])]
print(f"✓ Selected PCA components: {N_PCA_COMPONENTS}")

# ============================================================================
# STEP 3: FINAL CROSS-VALIDATION
# ============================================================================
print("\n" + "="*60)
print("STEP 3: FINAL CROSS-VALIDATION EVALUATION")
print("="*60)

eval_skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=0)
all_cv_probs_ct, all_cv_probs_sp, all_cv_probs_va, all_cv_labels = [], [], [], []
fold_preds_ct, fold_preds_sp, fold_preds_spatial = [], [], []
fold_thresholds_ct, fold_thresholds_sp, fold_thresholds_va = [], [], []

for fold_idx, (train_subj_idx, val_subj_idx) in enumerate(eval_skf.split(unique_subjs, y_unique)):
    print(f"--- Fold {fold_idx + 1}/{N_FOLDS} ---")
    train_mask, val_mask = np.isin(tr_subj_map, unique_subjs[train_subj_idx]), np.isin(tr_subj_map, unique_subjs[val_subj_idx])
    
    f_img_scaler = StandardScaler().fit(X_tr_slices[train_mask])
    f_pca = PCA(n_components=N_PCA_COMPONENTS, random_state=0, whiten=True).fit(f_img_scaler.transform(X_tr_slices[train_mask]))
    
    X_train_pca = f_pca.transform(f_img_scaler.transform(X_tr_slices[train_mask]))
    X_train_clin = X_tr_clin[train_mask]
    X_train_img = X_tr_slices[train_mask]
    y_train = y_tr_slices[train_mask]
    
    X_val_pca, X_val_clin, X_val_img, y_val = f_pca.transform(f_img_scaler.transform(X_tr_slices[val_mask])), X_tr_clin[val_mask], X_tr_slices[val_mask], y_tr_slices[val_mask]
    X_test_pca, X_test_clin, X_test_img = f_pca.transform(f_img_scaler.transform(X_te_slices)), X_te_clin, X_te_slices

    X_train_clin_t, X_train_pca_t = torch.tensor(X_train_clin, dtype=torch.float32).to(device), torch.tensor(X_train_pca, dtype=torch.float32).to(device)
    X_train_img_t = torch.tensor(X_train_img).view(-1, 1, 128, 128).float().to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    w = torch.where(y_train_t == 0, torch.tensor(NEG_WEIGHT, device=device), torch.tensor(SELECTED_POS_WEIGHT, device=device))
    
    # Train Models
    m_ct = ClinicalTransformer(n_inputs=X_tr_clin.shape[1]).to(device)
    opt_ct = optim.Adam(m_ct.parameters(), lr=1e-4)
    for _ in range(EPOCHS):
        m_ct.train(); opt_ct.zero_grad()
        loss = criterion(m_ct(X_train_clin_t + torch.randn_like(X_train_clin_t)*JITTER_STD), y_train_t, weight=w)
        loss.backward(); opt_ct.step()
    m_ct.eval(); [p.requires_grad_(False) for p in m_ct.parameters()]

    m_sp = ResidualSpectral(m_ct, SpectralViT(n_inputs=N_PCA_COMPONENTS)).to(device)
    opt_sp = optim.Adam(m_sp.m_res.parameters(), lr=1e-4)
    for _ in range(EPOCHS):
        m_sp.train(); opt_sp.zero_grad()
        loss = criterion(m_sp(X_train_pca_t + torch.randn_like(X_train_pca_t)*IMG_AUG_STD, X_train_clin_t), y_train_t, weight=w)
        loss.backward(); opt_sp.step()

    m_va = ResidualSpatial(m_ct, SpatialViT()).to(device)
    opt_va = optim.Adam(m_va.m_res.parameters(), lr=1e-4)
    for _ in range(EPOCHS):
        m_va.train(); opt_va.zero_grad()
        loss = criterion(m_va(X_train_img_t + torch.randn_like(X_train_img_t)*IMG_AUG_STD, X_train_clin_t), y_train_t, weight=w)
        loss.backward(); opt_va.step()

    # Evaluation
    with torch.no_grad():
        v_probs_ct, v_probs_sp, v_probs_va, v_labels = [], [], [], []
        for s in unique_subjs[val_subj_idx]:
            m = (tr_subj_map[val_mask] == s)
            v_probs_ct.append(m_ct(torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())
            v_probs_sp.append(m_sp(torch.tensor(X_val_pca[m], dtype=torch.float32).to(device), torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())
            v_probs_va.append(m_va(torch.tensor(X_val_img[m]).view(-1, 1, 128, 128).float().to(device), torch.tensor(X_val_clin[m], dtype=torch.float32).to(device)).mean().item())
            v_labels.append(y_val[m][0])
        
        fold_thresholds_ct.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_ct), np.array(v_labels), 'cpu'))
        fold_thresholds_sp.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_sp), np.array(v_labels), 'cpu'))
        fold_thresholds_va.append(calibrate_balanced(PassThrough(), None, np.array(v_probs_va), np.array(v_labels), 'cpu'))
        all_cv_probs_ct.extend(v_probs_ct); all_cv_probs_sp.extend(v_probs_sp); all_cv_probs_va.extend(v_probs_va); all_cv_labels.extend(v_labels)

        t_probs_ct, t_probs_sp, t_probs_va = [], [], []
        for s in np.unique(te_subj_map):
            m = (te_subj_map == s)
            t_probs_ct.append(m_ct(torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
            t_probs_sp.append(m_sp(torch.tensor(X_test_pca[m], dtype=torch.float32).to(device), torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
            t_probs_va.append(m_va(torch.tensor(X_test_img[m]).view(-1, 1, 128, 128).float().to(device), torch.tensor(X_test_clin[m], dtype=torch.float32).to(device)).mean().item())
        fold_preds_ct.append(t_probs_ct); fold_preds_sp.append(t_probs_sp); fold_preds_spatial.append(t_probs_va)

# Final calculation logic remains the same
th_ct, th_sp, th_va = np.mean(fold_thresholds_ct), np.mean(fold_thresholds_sp), np.mean(fold_thresholds_va)
y_test_labels = np.array([y_te_slices[te_subj_map == s][0] for s in np.unique(te_subj_map)])

# Aggregated Probabilities
cv_probs = {"Clinical": all_cv_probs_ct, "Spectral": all_cv_probs_sp, "Spatial": all_cv_probs_va}
test_probs = {
    "Clinical": np.mean(fold_preds_ct, axis=0),
    "Spectral": np.mean(fold_preds_sp, axis=0),
    "Spatial": np.mean(fold_preds_spatial, axis=0)
}
thresholds = {"Clinical": th_ct, "Spectral": th_sp, "Spatial": th_va}

def print_results_table(title, labels, prob_dict, thresh_dict):
    print(f"\n{title}")
    header = f"{'Model':<12} | {'AUC':<5} | {'AUPRC':<6} | {'Acc':<5} | {'B-Acc':<5} | {'Prec':<5} | {'Sens':<5} | {'Spec':<5} | {'F1':<5}"
    print("-" * len(header))
    print(header)
    print("-" * len(header))
    for name in ["Clinical", "Spectral", "Spatial"]:
        m = compute_comprehensive_metrics(np.array(labels), np.array(prob_dict[name]), thresh_dict[name])
        print(f"{name:<12} | {m['AUC']:.3f} | {m['AUPRC']:.3f} | {m['Acc']:.3f} | {m['B-Acc']:.3f} | {m['Prec']:.3f} | {m['Sens']:.3f} | {m['Spec']:.3f} | {m['F1']:.3f}")
    
    print("\nStatistical Significance (Incremental value over Clinical):")
    for name in ["Spectral", "Spatial"]:
        diff, p = compare_auc_significance(np.array(labels), np.array(prob_dict["Clinical"]), np.array(prob_dict[name]))
        sig = "*" if p < 0.05 else "n.s."
        print(f"  {name:<8} vs Clinical: ΔAUC {diff:+.3f}, p={p:.4f} ({sig})")

# Final Output
print_results_table("INTERNAL VALIDATION (5-Fold CV)", all_cv_labels, cv_probs, thresholds)
print_results_table("EXTERNAL TEST PERFORMANCE (CHH Dataset)", y_test_labels, test_probs, thresholds)

print("\n✓ Pipeline finished with comprehensive evaluation.")


Preparing MSW dataset (Forced 6-dim alignment) 
Pre-flight check: Validating MSW CSV mapping...
--- MSW CSV RAW MEANS ---
  > Age     : 62.13
  > Sex     : 0.26
  > Dur     : 8.47
  > LEDD    : 989.80
  > Off-Pre : 45.91
  > On-Pre  : 19.60

Final MSW Breakdown:
 - Unique Subjects on Disk: 111
 - Labeled Responders (1): 61
 - Labeled Non-Responders (0): 5
 - Unlabeled subjects (-1): 45
 - Verified Realized Means (Matched Data Only):
    > Age     : 63.08
    > Sex     : 0.26
    > Dur     : 8.44
    > LEDD    : 1003.95
    > Off-Pre : 45.62
    > On-Pre  : 19.55

❌ FULL MISSING LIST (45 subjects):
  [3, 4, 5, 8, 12, 13, 14, 17, 18, 21, 22, 24, 25, 27, 28, 31, 32, 34, 35, 37, 39, 40, 41, 42, 49, 50, 52, 54, 57, 61, 65, 67, 74, 76, 81, 82, 84, 88, 89, 94, 99, 101, 104, 105, 116]
Loaded cache with 7790 slices.

Preparing CHH dataset (Forced 6-dim alignment) 
Pre-flight check: Validating CHH CSV mapping...
--- CHH CSV RAW MEANS ---
  > Age     : 63.13
  > Sex     : 0.46
  > Dur     : 8.54